# Notebook Final - KNN 

Alur:
1. Baca `data/train.csv` dan `data/test.csv` (`dtype={"id": str}`).
2. Split validasi baku kelompok (seed 42, ~20% per kelas).
3. Evaluasi konfigurasi terpilih: **KNN k=21, weights=distance** (macro F1 validasi = 0.8318, terpilih dari `output/submission_kkn_decision_scores.csv`; hasil dari nol identik dengan scikit-learn).
4. Latih ulang prapemrosesan + model pada **seluruh** `train.csv`.
5. Simpan model + prapemroses ke `output/final/model/knn/`.
6. Muat ulang dari disk pada sesi bersih dan pastikan prediksi identik.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (confusion_matrix, f1_score,
                             precision_recall_fscore_support)

SEED = 42
BEST_K = 21
BEST_WEIGHTS = "distance"

REPO = Path.cwd()
while not (REPO / "data").exists():          # aman bila dijalankan dari src/...
    REPO = REPO.parent
MODEL_DIR = REPO / "output" / "final" / "model" / "knn"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPS = 1e-10
MISSING_TOKEN = "__MISSING__"
print("repo:", REPO)

repo: C:\Users\HP\Documents\IF3170_G03_FiveAgents


## 1. Implementasi dari nol (disalin agar notebook mandiri)

In [2]:
class KNNClassifier:
    """KNN biner (0/1) dari nol. Jarak Euclidean, prediksi per batch."""

    def __init__(self, n_neighbors=5, weights="uniform"):
        assert n_neighbors >= 1 and weights in ("uniform", "distance")
        self.n_neighbors = int(n_neighbors)
        self.weights = weights

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=np.float64)
        self.y_train = np.asarray(y).astype(np.int64).ravel()
        assert np.isfinite(self.X_train).all()
        return self

    def predict(self, X, batch_size=2000):
        X = np.asarray(X, dtype=np.float64)
        n = X.shape[0]
        out = np.empty(n, dtype=np.int64)
        k = min(self.n_neighbors, len(self.X_train))
        t2 = np.einsum("ij,ij->i", self.X_train, self.X_train)[None, :]
        for s in range(0, n, batch_size):
            xb = X[s:s + batch_size]
            d2 = np.einsum("ij,ij->i", xb, xb)[:, None] + t2 - 2.0 * (xb @ self.X_train.T)
            np.maximum(d2, 0.0, out=d2)
            near = np.argpartition(d2, kth=k - 1, axis=1)[:, :k]
            dist = np.sqrt(np.take_along_axis(d2, near, axis=1))
            lab = self.y_train[near]
            if self.weights == "uniform":
                s0 = (lab == 0).sum(axis=1); s1 = (lab == 1).sum(axis=1)
            else:
                w = 1.0 / (dist + EPS)
                s0 = np.where(lab == 0, w, 0.0).sum(axis=1)
                s1 = np.where(lab == 1, w, 0.0).sum(axis=1)
            out[s:s + batch_size] = (s1 > s0).astype(np.int64)  # seri -> 0
        return out


class KNNPreprocessor:

    def __init__(self, binarize_cols=("prior_cancellations", "prior_completed_bookings")):
        self.binarize_cols = tuple(binarize_cols)

    def fit(self, X):
        X = X.copy()
        self.num_cols = [c for c in X.columns if X[c].dtype != object]
        self.cat_cols = [c for c in X.columns if X[c].dtype == object]
        self.medians = {c: float(X[c].median()) for c in self.num_cols}
        self.thresholds = {c: float(X[c].median()) for c in self.num_cols
                           if c in self.binarize_cols}
        self.scale_cols = [c for c in self.num_cols if c not in self.thresholds]
        self.means, self.stds = {}, {}
        for c in self.scale_cols:
            col = X[c].fillna(self.medians[c])
            m, s = float(col.mean()), float(col.std(ddof=0))
            self.means[c] = m
            self.stds[c] = s if s > 0 and np.isfinite(s) else 1.0
        self.categories = {c: sorted(X[c].dropna().astype(str).unique().tolist())
                           for c in self.cat_cols}
        names = list(self.scale_cols) + list(self.thresholds)
        for c in self.cat_cols:
            names += [f"{c}={v}" for v in self.categories[c]] + [f"{c}={MISSING_TOKEN}"]
        self.feature_names_ = names
        return self

    def transform(self, X):
        X = X.copy()
        parts = []
        for c in self.scale_cols:
            col = X[c].fillna(self.medians[c]).to_numpy(dtype=np.float64)
            parts.append(((col - self.means[c]) / self.stds[c])[:, None])
        for c in self.thresholds:
            col = X[c].fillna(self.medians[c]).to_numpy(dtype=np.float64)
            parts.append((col > self.thresholds[c]).astype(np.float64)[:, None])
        for c in self.cat_cols:
            col = X[c].fillna(MISSING_TOKEN).astype(str).to_numpy()
            cats = self.categories[c]
            idx = {v: i for i, v in enumerate(cats)}
            blk = np.zeros((len(col), len(cats) + 1))
            for i, v in enumerate(col):
                blk[i, idx.get(v, len(cats))] = 1.0
            parts.append(blk)
        return np.hstack(parts)

    def to_dict(self):
        return {"version": 1, "binarize_cols": list(self.binarize_cols),
                "num_cols": self.num_cols, "cat_cols": self.cat_cols,
                "medians": self.medians, "thresholds": self.thresholds,
                "scale_cols": self.scale_cols, "means": self.means,
                "stds": self.stds, "categories": self.categories,
                "feature_names": self.feature_names_}

    @classmethod
    def from_dict(cls, d):
        o = cls(tuple(d["binarize_cols"]))
        for a in ("num_cols", "cat_cols", "scale_cols", "feature_names"):
            setattr(o, a[:-1] if a == "feature_names" else a, list(d[a]))
        o.feature_names_ = list(d["feature_names"])
        o.medians = {k: float(v) for k, v in d["medians"].items()}
        o.thresholds = {k: float(v) for k, v in d["thresholds"].items()}
        o.means = {k: float(v) for k, v in d["means"].items()}
        o.stds = {k: float(v) for k, v in d["stds"].items()}
        o.categories = {k: list(v) for k, v in d["categories"].items()}
        return o

print("kelas siap")

kelas siap


## 2. Baca data & split validasi baku

In [3]:
train = pd.read_csv(REPO / "data" / "train.csv", dtype={"id": str})
test = pd.read_csv(REPO / "data" / "test.csv", dtype={"id": str})

y = train["is_canceled"].to_numpy()
assert set(np.unique(y)) == {0, 1}

rng = np.random.default_rng(SEED)
valid_idx = np.sort(np.concatenate([
    rng.permutation(np.flatnonzero(y == lab))[:int(np.ceil(0.2 * (y == lab).sum()))]
    for lab in (0, 1)
]))
train_idx = np.setdiff1d(np.arange(len(train)), valid_idx)

X_raw = train.drop(columns=["id", "is_canceled"])
y_tr, y_va = y[train_idx], y[valid_idx]
print(f"train={len(train_idx)} {np.bincount(y_tr).tolist()} | "
      f"valid={len(valid_idx)} {np.bincount(y_va).tolist()}")

train=21898 [14723, 7175] | valid=5475 [3681, 1794]


## 3. Validasi konfigurasi terpilih

In [4]:
pre_v = KNNPreprocessor().fit(X_raw.iloc[train_idx])
Xv_tr = pre_v.transform(X_raw.iloc[train_idx])
Xv_va = pre_v.transform(X_raw.iloc[valid_idx])

m_v = KNNClassifier(n_neighbors=BEST_K, weights=BEST_WEIGHTS).fit(Xv_tr, y_tr)
pred_va = m_v.predict(Xv_va)

f1 = f1_score(y_va, pred_va, labels=[0, 1], average="macro", zero_division=0)
cm = confusion_matrix(y_va, pred_va, labels=[0, 1])
p, r, f, _ = precision_recall_fscore_support(y_va, pred_va, labels=[0, 1], zero_division=0)
print(f"macro F1 validasi = {f1:.4f}")
print("confusion matrix [0,1]:\n", cm)
print(f"kelas 0: P={p[0]:.4f} R={r[0]:.4f} F1={f[0]:.4f}")
print(f"kelas 1: P={p[1]:.4f} R={r[1]:.4f} F1={f[1]:.4f}")

macro F1 validasi = 0.8318
confusion matrix [0,1]:
 [[3357  324]
 [ 470 1324]]
kelas 0: P=0.8772 R=0.9120 F1=0.8942
kelas 1: P=0.8034 R=0.7380 F1=0.7693


## 4. Latih ulang pada SELURUH train.csv lalu simpan

In [5]:
pre = KNNPreprocessor().fit(X_raw)
X_full = pre.transform(X_raw)
model = KNNClassifier(n_neighbors=BEST_K, weights=BEST_WEIGHTS).fit(X_full, y)

np.savez_compressed(MODEL_DIR / "knn_model.npz",
                    X_train=model.X_train, y_train=model.y_train)
with open(MODEL_DIR / "preprocessing.json", "w") as fh:
    json.dump(pre.to_dict(), fh, indent=2)
with open(MODEL_DIR / "konfigurasi.json", "w") as fh:
    json.dump({"algorithm": "KNN from scratch", "n_neighbors": BEST_K,
               "weights": BEST_WEIGHTS, "metric": "euclidean", "seed": SEED,
               "macro_f1_validasi": round(float(f1), 6)}, fh, indent=2)
print("tersimpan di", MODEL_DIR)

tersimpan di C:\Users\HP\Documents\IF3170_G03_FiveAgents\output\final\model\knn


## 5. Muat ulang & uji prediksi identik

In [ ]:
with np.load(MODEL_DIR / "knn_model.npz") as z:
    model2 = KNNClassifier(BEST_K, BEST_WEIGHTS).fit(z["X_train"], z["y_train"])
with open(MODEL_DIR / "preprocessing.json") as fh:
    pre2 = KNNPreprocessor.from_dict(json.load(fh))

X_test = pre2.transform(test.drop(columns=["id"]))
pred_test = model2.predict(X_test)

assert np.array_equal(model2.predict(X_test[:500]), model.predict(X_test[:500])), "prediksi model berbeda!"
assert np.array_equal(pre2.transform(X_raw.iloc[:500]), X_full[:500]), "prapemroses berbeda!"
assert set(np.unique(pred_test)) <= {0, 1}
print("uji load ulang: prediksi identik ✔ | pred test:",
      np.bincount(pred_test, minlength=2).tolist())

uji load ulang: prediksi identik ✔ | pred test: [4820, 2024]
